# ***HTM Trainer:***

In [ ]:
import sys, subprocess
sys.path.insert(0, subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).decode().strip())

In [ ]:
import os

# from tensorflow.python.saved_model.pywrap_saved_model.constants import SAVED_MODEL_FILENAME_PB

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import torch
from tqdm import tqdm
tqdm.pandas()
%matplotlib inline

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='htm')
torch.manual_seed(42)

from dataloaders import get_replicated_dataloader
from models import CNNModel, CNNTrainer
from utils import generate_config, save_htm_model, get_htm_model

In [ ]:
# CONSTANTS
exp_number = 13
image_size = 128
dataset_name = 'celeba_stripes'
sample_indices = list(range(0, 480, 1))


# HTM EPOCH COUNTERS
SAVED_HTM_EPOCH = 20
STEP_SIZE = 1

# Configuration:

In [ ]:
model_path = f'e{exp_number}_{dataset_name}_s{image_size}'

# Generate config with all computed dimensions
config = generate_config(
    image_size=image_size,
    base_enc=4,
    split_factor=4,
    channels=1,
    data_path=f'data/{dataset_name}',
    cnn={"inner_dim": 512, "learning_rate": 0.001, "loss_weights": {"mse": 0.10, "ssim": 0.8, "tv": 0.10}},
    training={"batch_size": 256, "total_size": 9600, "patience": 10},

    save_path=model_path,
)

project_root = os.getcwd().replace('notebooks', 'configs')
config_path = os.path.join(project_root, model_path + ".json")
config.save(path=config_path)

# Dataloader for celeba from reprocessed dataset

#### - Creating subset from celeba - filtered and sorted by attributes and head tilt. ~5000 samples.
#### - Compositing images - from groups of 4 - generating a composition of all possible combinations. ~3200 samples.
#### - Initiating Special Dataloader - replicated k images n times with random masks. k*n samples.
            - 10 groups sampled from 40 original images.

            - 480 composite samples generated from 24 combinations for each group, vertical and horizonal.

            - 20 different masks applied on each sample.

            --> Total size - 9600 samples.
#### - Dataloader transformation: 64*64 resized, centered, grayscale images.

In [ ]:
# Initiate celeba Dataloader From subset of Images
dataloader = get_replicated_dataloader(
  dataset_name='celeba',
  sample_indices=sample_indices,
  total_size=config.training['total_size'],
  download=True,
  image_size=config.image['size'],
  root_dir=config.runtime['data_path'],
  batch_size=config.training["batch_size"]
)

dataloader.dataset.return_idx = True

___
___


# ***Model:***

#### HTM Visualization tools - View learned SDRs, overlapping between them and impact on result.

In [ ]:
from utils import HTMAnalyzer
from htm.bindings.sdr import SDR

def to_sdr(t):
    """Convert tensor to SDR if needed."""
    if isinstance(t, torch.Tensor):
        arr = t.cpu().numpy()
        sdr = SDR(arr.shape)
        sdr.dense = arr
        return sdr
    return t

def visualize_htm_state(with_cnn_results = False, htm_model = None, model = None):
    if (htm_model is None or model is None):
        print("is None")
        return
    htm_model.learning_enabled = False

    # Get n outputs from your model
    analyzer = HTMAnalyzer()

    batch_tensors = model.generate_sdr(10, htm_model, rand_mask=True)
    batch = [to_sdr(t) for t in batch_tensors]

    # metrics = analyzer.track_sdr_sequence(batch, name="gen_analysis")
    analyzer.plot_overlap_matrix(batch)

    # Generate and plot sample images using the trained model
    generated_samples = model.generate_from_SDR(batch_tensors)
    analyzer.plot_sdrs_grid(batch, ncols=5)
    if with_cnn_results:
        model.plot_images(generated_samples)

    htm_model.learning_enabled = True


## Initiate Pipeline:
##### - Load Trained HTM Model or Cached HTM Model by Config. If neither exist - initiate new HTM.
##### - CNN model and Trainer initiated.
##### - HTM training is part of the Trainer main loop.

In [ ]:
# Check if trained HTM model exists
htm_model_path = f"../results/{model_path}/trained_htm_epoch_{SAVED_HTM_EPOCH}.pkl"
htm_model = get_htm_model(htm_model_path, config)

In [ ]:
# Initialize CNN model from config
model = CNNModel(config=config)

In [ ]:
# Initialize CNN trainer with HTM preprocessor and config
trainer = CNNTrainer(model=model, htm_model=htm_model, config=config)

In [ ]:
visualize_htm_state(model=model, htm_model=htm_model)

___
___

## Train HTM:

#### 0-10

In [ ]:
# # Train htm model STEP_SIZE epochs
# htm_model.learning_enabled = True
#
# trainer.train(dataloader=dataloader, epochs=[SAVED_HTM_EPOCH, SAVED_HTM_EPOCH + STEP_SIZE], save_path=config.runtime['save_path'],
#               idx_htm_only=SAVED_HTM_EPOCH)
#
# # update current htm epoch:
# SAVED_HTM_EPOCH += STEP_SIZE
#
# # Save the trained HTM model
# save_htm_model(htm_model, save_path=f"../results/{config.runtime['save_path']}", model_name=f"trained_htm_epoch_s128_{SAVED_HTM_EPOCH}")

#### Train HTM

Save model every epoch

In [ ]:
htm_model.learning_enabled = True
save_path=f"../results/{config.runtime['save_path']}"

last_saved_htm = SAVED_HTM_EPOCH

# Save inner states until 30 epochs learnt
for i in range(SAVED_HTM_EPOCH, 30):
    model_name=f"trained_htm_epoch_{i + 1}"
    print(f'{'='*80}\nTrain HTM: [{i}:{i + 1}]')

    # train:
    trainer.train(dataloader=dataloader, epochs=[i, i + 1], save_path=config.runtime['save_path'],
                  idx_htm_only=i+1)

    # save:
    save_htm_model(htm_model, save_path, model_name)
    last_saved_htm = i+1

# update htm state:
SAVED_HTM_EPOCH = last_saved_htm



In [ ]:
visualize_htm_state(model=model, htm_model=htm_model)

## Train CNN:

In [ ]:
# Train CNN
htm_model.learning_enabled = False
trainer.train(dataloader=dataloader, epochs=[0,50], save_path=config.runtime['save_path'] + 'refined')

In [ ]:
visualize_htm_state(model=model, htm_model=htm_model, with_cnn_results=True)

# Inference:
#### - Generate images from random binary matrix.

In [ ]:
# Generate and plot sample images using the trained model
generated_images = model.generate(num_samples=1, htm_model=htm_model)
model.plot_images(generated_images)

In [ ]:

# Generate and plot sample images using the trained model
generated_images = model.generate(num_samples=25, htm_model=htm_model)
model.plot_images(generated_images)

# Load cnn trained model

In [ ]:
# 1. Create model with same config
model = CNNModel(config)

# 2. Load the check_point
check_point = torch.load("../results/e13_celeba_stripes_s128_wo/best_model.pt", weights_only=False, map_location=model.device)

# 3. Load weights into model.
model.load_state_dict(check_point['model_state'])

# 4. Initialize Trainer with CNN model, HTM model and configuration.
trainer = CNNTrainer(
    model=model,
    htm_model=htm_model,
    config=config
)

# 5. Generate images
images = model.generate(num_samples=5, htm_model=htm_model)